In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16, VGG19, InceptionV3, MobileNetV2, DenseNet121


ModuleNotFoundError: No module named 'cv2'

In [ ]:
# Load images for training and validation.
sdir = '/content/drive/MyDrive/boroi dataset'
slist = os.listdir(sdir)
classes = []
filepaths = []
labels = []
for d in slist:
    dpath = os.path.join(sdir, d)
    if os.path.isdir(dpath):
        classes.append(d)
class_count = len(classes)
for klass in classes:
    classpath = os.path.join(sdir, klass)
    filelist = os.listdir(classpath)
    for f in filelist:
        fpath = os.path.join(classpath, f)
        filepaths.append(fpath)
        labels.append(klass)

df = pd.DataFrame({'filepaths': filepaths, 'labels': labels})

In [ ]:
from PIL import Image
import os

def make_square(image_path, save_path):
    """Crops an image to a square shape and saves it.

    Args:
        image_path (str): Path to the input image.
        save_path (str): Path to save the cropped image.
    """
    img = Image.open(image_path)
    width, height = img.size

    # Find the smaller side
    min_dim = min(width, height)

    # Calculate cropping box
    left = (width - min_dim) / 2
    top = (height - min_dim) / 2
    right = (width + min_dim) / 2
    bottom = (height + min_dim) / 2

    # Crop and save
    img_cropped = img.crop((left, top, right, bottom))
    img_cropped.save(save_path)

# Example use
# Providing the image path directly, since __file__ is not available in interactive sessions
image_path_path = "/content/drive/MyDrive/boroi dataset"  # Replace with your actual image path
# Make sure the file exists at this location and the path is correct.

# Check if Google Drive is mounted:
from google.colab import drive
drive.mount('/content/drive')

# Verify the file exists:
import os
# Loop through all files in the directory
for filename in os.listdir(image_path_path):
    # Check if the file is an image
    if filename.endswith(('.jpg', '.jpeg', '.png')): # Add other image extensions if needed
        # Construct the full path to the image
        image_path = os.path.join(image_path_path, filename)

        # Construct the save path
        save_path = os.path.join(image_path_path, "squared_" + filename)  # Or any desired naming scheme

        # Apply the make_square function to the image
        make_square(image_path, save_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Split dataset into training, validation, and test sets.
train_split = 0.8
test_split = 0.1
dummy_split = test_split / (1 - train_split)
train_df, dummy_df = train_test_split(df, train_size=train_split, shuffle=True, random_state=125)
test_df, valid_df = train_test_split(dummy_df, train_size=dummy_split, shuffle=True, random_state=125)

In [ ]:
# Data preprocessing and augmentation.
batch_size = 16
def scalar(x):
    return x / 127.5 - 1

trgen = tf.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=scalar, horizontal_flip=True)
train_gen = trgen.flow_from_dataframe(train_df, x_col='filepaths', y_col='labels', target_size=(224,224), class_mode='categorical', batch_size=batch_size, shuffle=True, seed=123)

tvgen = tf.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=scalar)
valid_gen = tvgen.flow_from_dataframe(valid_df, x_col='filepaths', y_col='labels', target_size=(224,224), class_mode='categorical', batch_size=batch_size, shuffle=False)
test_gen = tvgen.flow_from_dataframe(test_df, x_col='filepaths', y_col='labels', target_size=(224,224), class_mode='categorical', batch_size=batch_size, shuffle=False)
test_labels = test_gen.labels

Found 1889 validated image filenames belonging to 5 classes.
Found 237 validated image filenames belonging to 5 classes.
Found 236 validated image filenames belonging to 5 classes.


In [ ]:
# Define a function to create models.
def create_model(base_model):
    base_model.trainable = False  # Freeze the base model layers
    inputs = keras.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    outputs = keras.layers.Dense(class_count, activation='softmax')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
# Models list (VGG16, VGG19, InceptionV3, MobileNetV2, DenseNet121)
from tensorflow.keras.applications import VGG16, VGG19, InceptionV3, MobileNetV2, DenseNet121

models = {
    "VGG16": create_model(VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))),

}

In [ ]:
# Train models (pre-trained phase)
histories = {}
for name, model in models.items():
    print(f"Training {name} (pre-trained)...")
    histories[name] = model.fit(train_gen, validation_data=valid_gen, epochs=10, verbose=1)


Training VGG16 (pre-trained)...


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 1687s 14s/step - accuracy: 0.1756 - loss: 1.7046 - val_accuracy: 0.2278 - val_loss: 1.4936
Epoch 2/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 1698s 14s/step - accuracy: 0.3900 - loss: 1.4340 - val_accuracy: 0.4346 - val_loss: 1.3871
Epoch 3/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 1666s 14s/step - accuracy: 0.4006 - loss: 1.3998 - val_accuracy: 0.4346 - val_loss: 1.3582
Epoch 4/10
 72/119 ━━━━━━━━━━━━━━━━━━━━ 9:46 12s/step - accuracy: 0.4429 - loss: 1.3402

In [ ]:
# Fine-tuning process: Unfreeze base model layers and train again.
def fine_tune_model(model, base_model):
    # Unfreeze the base model layers
    base_model.trainable = True

    # It's recommended to unfreeze the last few layers only (e.g., 50 layers)
    for layer in base_model.layers[:-50]:
        layer.trainable = False

    model.compile(optimizer=Adam(learning_rate=0.00001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
# Fine-tune the models
fine_histories = {}
for name, model in models.items():
    print(f"Fine-tuning {name}...")
    fine_model = fine_tune_model(model, model.layers[1])  # model.layers[1] is the base model
    fine_histories[name] = fine_model.fit(train_gen, validation_data=valid_gen, epochs=10, initial_epoch=histories[name].epoch[-1], verbose=1)

In [ ]:
# Evaluate models after fine-tuning
results = {}
for name, model in models.items():
    preds = model.predict(test_gen, verbose=1)
    y_pred = np.argmax(preds, axis=1)
    print(f"Results for {name} after fine-tuning:")
    cm = confusion_matrix(test_labels, y_pred)
    print(cm)
    print(classification_report(test_labels, y_pred, target_names=list(test_gen.class_indices.keys())))
    results[name] = cm
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=list(test_gen.class_indices.keys()), yticklabels=list(test_gen.class_indices.keys()))
    plt.title(f'Confusion Matrix - {name}')
    plt.show()

In [ ]:
# Evaluate models after fine-tuning
results = {}
for name, model in models.items():
    preds = model.predict(test_gen, verbose=1)
    y_pred = np.argmax(preds, axis=1)
    print(f"Results for {name} after fine-tuning:")
    cm = confusion_matrix(test_labels, y_pred)
    print(cm)
    print(classification_report(test_labels, y_pred, target_names=list(test_gen.class_indices.keys())))
    results[name] = cm
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=list(test_gen.class_indices.keys()), yticklabels=list(test_gen.class_indices.keys()))
    plt.title(f'Confusion Matrix - {name}')
    plt.show()

In [ ]:

# Assuming histories and fine_histories are dictionaries containing the pre-training and fine-tuning histories.
data = []

for name, history in fine_histories.items():
    pre_train_epochs = len(histories[name].history['val_accuracy'])
    epochs = range(pre_train_epochs + 1, pre_train_epochs + len(history.history['val_accuracy']) + 1)

    for epoch, val_acc in zip(epochs, history.history['val_accuracy']):
        data.append({'Epoch': epoch, 'Validation Accuracy': val_acc, 'Model': f'{name} Fine-Tuned'})

# Convert data to DataFrame
df = pd.DataFrame(data)

# Plot using Seaborn
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='Epoch', y='Validation Accuracy', hue='Model', marker='o')

plt.xlabel("Epochs")
plt.ylabel("Validation Accuracy")
plt.title("Fine-Tuned Validation Accuracy Comparison")
plt.grid(True)
plt.legend(title="Model")
plt.show()



